In [ ]:
def main(datasource, start_date, end_date):
    """
    factor function

    Args:
        datasource (str): Datasource table name
        start_date (str): Start date in 'YYYY-MM-DD HH:MM:SS' format
        end_date (str): End date in 'YYYY-MM-DD HH:MM:SS' format

    Returns:
        pd.DataFrame: Factor data with columns ['date', 'instrument', 'factor']
    """
    import pandas as pd
    import dai

    # 因子公式 https://bigquant.com/wiki/doc/Rceb2JQBdS
    sql = f"""
    WITH cte_snapshot AS (
        SELECT
            date, instrument_id, price, volume, 

            -- 交易日
            strftime(date, '%Y-%m-%d') as trading_day,

            -- 计算中间价格
            (ask_price1 + bid_price1) / 2 as mid_price,

            -- 计算 加权5档买方量
            (
                COALESCE(bid_volume1, 0) * 1.0 + 
                COALESCE(bid_volume2, 0) * EXP(-0.3) + 
                COALESCE(bid_volume3, 0) * EXP(-0.6) + 
                COALESCE(bid_volume4, 0) * EXP(-0.9) + 
                COALESCE(bid_volume5, 0) * EXP(-1.2)
            ) as weight_bid,

            -- 计算 加权5档卖方量
            (
                COALESCE(ask_volume1, 0) * 1.0 + 
                COALESCE(ask_volume2, 0) * EXP(-0.3) + 
                COALESCE(ask_volume3, 0) * EXP(-0.6) + 
                COALESCE(ask_volume4, 0) * EXP(-0.9) + 
                COALESCE(ask_volume5, 0) * EXP(-1.2)
            ) as weight_ask,

            -- 计算 加权订单簿不平衡度
            (weight_bid - weight_ask) / (weight_bid + weight_ask + 1e-8) as weighted_imbalance,

            -- 计算 价差因子
            (ask_price1 - bid_price1) / mid_price as relative_spread,

            -- 计算 3档买方量
            (
                COALESCE(bid_volume1, 0) + 
                COALESCE(bid_volume2, 0) + 
                COALESCE(bid_volume3, 0)
            ) as total_bid,

            -- 计算 3档卖方量
            (
                -- 加权卖方成交量
                COALESCE(ask_volume1, 0) + 
                COALESCE(ask_volume2, 0) + 
                COALESCE(ask_volume3, 0)
            ) as total_ask,

            -- 计算 深度比率
            (total_bid - total_ask) / (total_bid + total_ask + 1e-8) as depth_ratio,

            -- 计算 压力因子
            (weighted_imbalance / (sqrt(abs(relative_spread)) + 1e-8)) * weighted_imbalance as raw_pressure,

            -- 按15分钟分段的时间列
            CASE 
                -- 上午时间段
                WHEN strftime(date, '%H%M') >= '0930' AND strftime(date, '%H%M') < '0945' THEN 94500
                WHEN strftime(date, '%H%M') >= '0945' AND strftime(date, '%H%M') < '1000' THEN 100000
                WHEN strftime(date, '%H%M') >= '1000' AND strftime(date, '%H%M') < '1015' THEN 101500
                WHEN strftime(date, '%H%M') >= '1015' AND strftime(date, '%H%M') < '1030' THEN 103000
                WHEN strftime(date, '%H%M') >= '1030' AND strftime(date, '%H%M') < '1045' THEN 104500
                WHEN strftime(date, '%H%M') >= '1045' AND strftime(date, '%H%M') < '1100' THEN 110000
                WHEN strftime(date, '%H%M') >= '1100' AND strftime(date, '%H%M') < '1115' THEN 111500
                WHEN strftime(date, '%H%M') >= '1115' AND strftime(date, '%H%M') <= '1130' THEN 113000
                
                -- 下午时间段
                WHEN strftime(date, '%H%M') >= '1300' AND strftime(date, '%H%M') < '1315' THEN 131500
                WHEN strftime(date, '%H%M') >= '1315' AND strftime(date, '%H%M') < '1330' THEN 133000
                WHEN strftime(date, '%H%M') >= '1330' AND strftime(date, '%H%M') < '1345' THEN 134500
                WHEN strftime(date, '%H%M') >= '1345' AND strftime(date, '%H%M') < '1400' THEN 140000
                WHEN strftime(date, '%H%M') >= '1400' AND strftime(date, '%H%M') < '1415' THEN 141500
                WHEN strftime(date, '%H%M') >= '1415' AND strftime(date, '%H%M') < '1430' THEN 143000
                WHEN strftime(date, '%H%M') >= '1430' AND strftime(date, '%H%M') < '1445' THEN 144500
                WHEN strftime(date, '%H%M') >= '1445' AND strftime(date, '%H%M') < '1457' THEN 150000
                
                -- 其他时间段（如果有数据）
                ELSE -1
            END as time_segment

        FROM {datasource}
        WHERE time_segment != -1
    ),
    -- 计算窗口直播
    cte_rolling AS(
        SELECT 
            *,
            lag(mid_price, 1) OVER (PARTITION BY instrument_id, trading_day, time_segment ORDER BY date) as prev_mid_price,
            abs(mid_price / prev_mid_price - 1) as returns,
        FROM cte_snapshot
    ),
    -- 计算分组因子值
    cte_window AS (
        SELECT
            trading_day, time_segment, instrument_id,

            -- 标准化原始压力因子
            avg(raw_pressure) as raw_pressure_mean,
            nanstd(raw_pressure) as raw_pressure_std,
            (last(raw_pressure) - raw_pressure_mean) / raw_pressure_std as standardized_pressure,

            -- 计算波动率阀值
            CASE
                WHEN COUNT(*) > 10
                THEN quantile(returns, 0.8)
                ELSE 0.01
            END as volatility_threshold,

            -- 根据波动率调整因子（高波动时降低因子值）
            nanstd(log(mid_price / prev_mid_price)) as volatility,
            CASE 
                WHEN volatility > volatility_threshold
                THEN standardized_pressure * 0.7
                ELSE standardized_pressure
            END as adjusted_pressure,

            tanh(adjusted_pressure) as factor
        FROM cte_rolling
        GROUP BY instrument_id, trading_day, time_segment
        ORDER BY instrument_id, time_segment
    )
    -- 映射instrument_id到instrument列
    SELECT
        -- 转换为15分钟的date列
        CAST(CONCAT(
            f.trading_day,
            ' ',
            strftime(strptime(LPAD(f.time_segment, 6, '0'), '%H%M%S'), '%H:%M:%S')
        ) AS DATETIME) AS date,
        all_instruments.instrument,
        -- 因子方向为 -1
        f.factor * -1 as factor
    FROM cte_window f
    LEFT JOIN all_instruments USING (instrument_id)
    """

    df = dai.query(sql, filters={'date': [start_date, end_date]}).df()
    return df


if __name__ == '__main__':
    """
    FOR Development in __main__

    This section demonstrates how to:
    1. Calculate factors using the main() function
    2. Visualize and analyze results using the factorlens module

    Tips:
        - Adjust the date range to test different time periods
        - Modify the SQL query in main() to create your own factors
        - Use factorlens to evaluate factor performance metrics
    """
    from bigmodule import M
    from datetime import datetime
    import structlog

    logger = structlog.get_logger()

    datasource = 'cpt_dwc_2026_stock_hs300_snapshot'
    start_date = '2023-01-01 00:00:00'
    end_date = '2024-12-31 23:59:59'

    logger.info(f"Calculating factor for period: {start_date} to {end_date}")
    t1 = datetime.now()
    data = main(datasource, start_date, end_date)
    t2 = datetime.now()
    total_seconds = (t2 - t1).total_seconds()
    minutes = int(total_seconds // 60)
    seconds = int(total_seconds % 60)
    logger.info(f"Time spent: {minutes}min{seconds}s")
    logger.info(f"Factor data shape: {data.shape}")
    logger.info(f"\nSample data:\n{data.head()}")

    # 因子评估
    results = M.eval_dwc._latest(
        data=data
    )